# Introduction:
This script will measure factors that contribute to job satisfaction dependent on hours worked through analysis of 18 waves of British Household Panel Data, which is excellent data in capturing social-economic factors throughout a relatively long period of mostly the same individuals.

---------------------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Dataset: British Household Panel Survey 
Description: The data is combining all the waves of the British Household Panel Survey (wave 1 to wave 18).  

| **Variable**                                | **Variable Name in the Data File** | **Description of the Variable**                                                                                       |
|---------------------------------------------|-----------------------------------|------------------------------------------------------------------------------------------------------------------------|
| Personal identification number              | pid                               | Identification used to identify individuals in the data                                                                |
| Wave number                                 | wave                              | 1 to 18                                                                                                                |
| Real (1987=100) annual family income        | rfinc                              | In real GBP                                                                                                            |
| Sex of the respondent                       | female                            | female = 1, male = 0                                                                                                   |
| Hourly Wage                                 | hourlypay                         | In real GBP                                                                                                            |
| Age at date of interview                    | age                               | Number of years                                                                                                        |
| Years of tenure at current job              | ten                               | Number of years                                                                                                        |
| Job satisfaction: total pay                 | jbsat2x                           | 1: no satisfaction, 2: very dissatisfied, 3: dissatisfied, 4: neither satisfied nor dissatisfied, 5: satisfied, 6: very satisfied, 7: completely satisfied |
| Job satisfaction: security                  | jbsat4x                           | 1: no satisfaction, 2: very dissatisfied, 3: dissatisfied, 4: neither satisfied nor dissatisfied, 5: satisfied, 6: very satisfied, 7: completely satisfied |
| Job satisfaction: work itself               | jbsat6x                           | 1: no satisfaction, 2: very dissatisfied, 3: dissatisfied, 4: neither satisfied nor dissatisfied, 5: satisfied, 6: very satisfied, 7: completely satisfied |
| Job satisfaction: hours worked              | jbsat7x (picked)                  | 1: no satisfaction, 2: very dissatisfied, 3: dissatisfied, 4: neither satisfied nor dissatisfied, 5: satisfied, 6: very satisfied, 7: completely satisfied |
| Job satisfaction: overall                   | jbsat                             | 1: no satisfaction, 2: very dissatisfied, 3: dissatisfied, 4: neither satisfied nor dissatisfied, 5: satisfied, 6: very satisfied, 7: completely satisfied |
| Individual in a Union Job                   | union                             | In a union job = 1, otherwise = 0                                                                                        |
| Individual is a Union Member                | unionm                            | Union Member = 1, otherwise = 0                                                                                         |
| Individual has job opportunities at employer| jobop                             | Yes = 1, no = 0                                                                                                        |
| Establishment size less than 25 employees (small) | sest                             | Yes = 1, No = 0                                                                                                        |
| Establishment size between 25 and 99 employees (medium) | mest                             | Yes = 1, No = 0                                                                                                        |
| Establishment size 100 or more employees (large) | lest                             | Yes = 1, No = 0                                                                                                        |


---------------------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:

#summarise with pandas
import pandas as pd

df = pd.read_stata('/<your_file_path>/project5.dta')
print(df.head())

num_obs = len(df)

print(num_obs)


        pid  wave         rfinc jbhrs jbsat2x                  jbsat4x  \
0  10007857     2  19731.597656    54       5  not satis/dissat          
1  10014608     1  40067.777344    40       6                        6   
2  10014608     2  43806.582031    42       6                        6   
3  10014608     3  42772.035156    40       6                        6   
4  10014608     5  56064.785156    42       6                        6   

                   jbsat6x                  jbsat7x jbsat age  ...  reg11x  \
0                        6                        3     5  59  ...     0.0   
1  completely satis         completely satis            6  57  ...     0.0   
2                        6                        6     6  58  ...     0.0   
3                        6                        6     6  59  ...     0.0   
4                        6                        6     6  61  ...     0.0   

   reg12x  hourlypay    logpay  female  sest  mest  lest  npermj  parth  
0     0.0   

---------------------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Stata script:
```text 
*FE=fixed effect
*RE=random effect

* Describe the dataset
describe

* Set the panel data structure with 'pid' as the panel identifier and 'wave' as the time variable
xtset pid wave

* Describe the panel structure
xtdescribe

* Summarise the variables for panel data analysis
xtsum

* Generate squared variables for non-linear effects in the model
gen ten2 = ten^2   // Square of 'ten' (tenure)
gen age2 = age^2   // Square of 'age'

* Run a RE regression with robust standard errors
xtreg jbsat7x hourlypay ten ten2 age age2, re robust

* Run a FE regression with robust standard errors
xtreg jbsat7x hourlypay ten ten2 age age2, fe robust

* Run a FE regression without robust standard errors
xtreg jbsat7x hourlypay ten ten2 age age2, fe

* Store the FE estimates
estimates store FE

* Run a RE regression without robust standard errors
xtreg jbsat7x hourlypay ten ten2 age age2, re

* Store the RE estimates
estimates store RE

* Perform a Hausman test to compare the fixed vs RE models
hausman FE RE

* Run a FE regression with additional regressor variables 'reg1x' through 'reg12x'
xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x, fe robust

* Perform a joint test to check the significance of the 'reg1x' through 'reg12x' variables
test reg1x reg2x reg3x reg4x reg5x reg6x reg7x reg8x reg9x reg10x reg11x reg12x

* Run a FE regression with additional social variables 'soc1x' through 'soc9x'
xtreg jbsat7x hourlypay ten ten2 age age2 soc1x-soc9x, fe robust

* Perform a joint test to check the significance of the 'soc1x' through 'soc9x' variables
test soc1x soc2x soc3x soc4x soc5x soc6x soc7x soc8x soc9x

* Run separate FE regressions for males (female == 0) and females (female == 1) with all regressors
xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x soc1x-soc9x if female==0, fe robust
xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x soc1x-soc9x if female==1, fe robust



---------------------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Full Stata log:

```text
. *Load the dataset in Stata
. clear

. use project5, clear

. *Visualise the data
. describe

Contains data from project5.dta
 Observations:        94,382                  
    Variables:            43                  

Variable      Storage   Display    Value
    name         type    format    label      Variable label

pid	}cross-wave person identifier
wave	}Wave Number Variable
rfinc	}Real (1987=100) Family Income: Labour+Non-Labour
jbhrs	}no. of hours normally worked per week
jbsat2x	}job satisfaction: total pay
jbsat4x	}job satisfaction: security
jbsat6x	}job satisfaction: work itself
jbsat7x	}job satisfaction: hours worked
jbsat	}job satisfaction: overall
age	}age at date of interview
union	}DV=1 if individual in a union job
unionm	}DV=1 if individual is a union member
jobop	}DV=1 if Individual has job opportunities at employer
ten	}Years of Tenure on Current Job
soc1x	}DV=1 if Manager or Administrator Occupation
soc2x	}DV=1 if Professional Occupation
soc3x	}DV=1 if Associate Professional or Technical Occupation
soc4x	}DV=1 if Clerical Occupation
soc5x	}DV=1 if Craft & Related Occupation
soc6x	}DV=1 if Personal & Protective Services Occupation
soc7x	}DV=1 if Sales Occupation
soc8x	}DV=1 if Plant & Machine Operative Occupation
soc9x	}DV=1 if Other Occupation
reg1x	}DV=1 if region is London
reg2x	}DV=1 if region is South East
reg3x	}DV=1 if region is South West
reg4x	}DV=1 if region is East of England
reg5x	}DV=1 if region is East Midlands
reg6x	}DV=1 if region is West Midlands
reg7x	}DV=1 if region is North West
reg8x	}DV=1 if region is Yorkshire & the Humber
reg9x	}DV=1 if region is North East
reg10x	}DV=1 if region is Wales
reg11x	}DV=1 if region is Scotland
reg12x	}DV=1 if region is Northern Ireland
hourlypay	}Gross Hourly Pay
logpay	}Natural log of the gross hourly pay
female	}DV=1 if gender is female
sest	}DV=1 if establishment size less than 25 employees
mest	}DV=1 if establishment size between 25-99 employees
lest	}DV=1 if establishment size 100 or more employees
npermj	}DV=1 if individual has a non-permanent job
parth	}DV=1 if Part-time Worker, Hours<=30

Sorted by: pid  wave

. * In our data cross-sectional identifier is pid and time identifier is wave
. * Allows use of panel commands and some time series operators
. xtset pid wave


}}
Delta: 1 unit

. xtdescribe

     pid:  10007857, 10014608, ..., 1.871e+08                n =      15958
    wave:  1, 2, ..., 18                                     T =         18
           Delta(wave) = 1 unit
           Span(wave)  = 18 periods
           (pid*wave uniquely identifies each observation)

Distribution of T_i:   min      5%     25%       50%       75%     95%     max
                         1       1       2         4         9      17      18

Freq.  Percent    Cum.   Pattern
 
      592      3.71    3.71   1.................
      532      3.33    7.04   111111111111111111
      420      2.63    9.68   ........1111111111
      394      2.47   12.14   ........1.........
      291      1.82   13.97   11................
      275      1.72   15.69   ......11111.......
      271      1.70   17.39   .................1
      250      1.57   18.96   .........1........
      232      1.45   20.41   ..........1.......
    12701     79.59  100.00  (other patterns)
 
    15958    100.00           XXXXXXXXXXXXXXXXXX


. xtsum

Variable               Mean   Std. dev.       Min        Max     Observations

pidoverall   4.11e+07   3.96e+07   1.00e+07   1.87e+08 N =   94382
between  4.64e+07   1.00e+07   1.87e+08 n =   15958
within          0   4.11e+07   4.11e+07  T-bar =  5.9144

waveoverall   10.01067   4.950013          1         18 N =   94382
between  4.461133          1         18 n =   15958
within   3.660741  -2.239331   22.51067  T-bar =  5.9144

rfincoverall   19642.06   11615.78     -9.375   664759.9 N =   94382
between  10055.74          0   167784.6 n =   15958
within   7228.341  -106146.8   618773.8  T-bar =  5.9144

jbhrsoverall    33.6658   10.74073          1         80 N =   94382
between  10.71371          1         80 n =   15958
within   5.895518  -9.795734   76.95992  T-bar =  5.9144

jbsat2xoverall   4.856032   1.565966          1          7 N =   94382
between  1.301481          1          7 n =   15958
within    1.14979  -.6733799   9.668532  T-bar =  5.9144

jbsat4xoverall   5.396156   1.512004          1          7 N =   94382
between  1.231742          1          7 n =   15958
within   1.132102  -.2705106   10.98439  T-bar =  5.9144

jbsat6xoverall    5.45336   1.334389          1          7 N =   94382
between  1.091616          1          7 n =   15958
within   .9950341  -.0466402   10.12003  T-bar =  5.9144

jbsat7xoverall   5.222553   1.438601          1          7 N =   94382
between  1.161169          1          7 n =   15958
within    1.05562  -.1941136   10.00033  T-bar =  5.9144

jbsatoverall   5.371469   1.299105          1          7 N =   94382
between  1.048899          1          7 n =   15958
within   .9798218  -.1285309   10.27147  T-bar =  5.9144

ageoverall   37.63339   11.85272         16         65 N =   94382
between  12.70312         16         65 n =   15958
within   3.672936   25.38339   49.96673  T-bar =  5.9144

unionoverall   .5111458   .4998785          0          1 N =   90258
between  .4427572          0          1 n =   15499
within   .2688601  -.4332986    1.45559  T-bar = 5.82347

unionmoverall   .4282487   .4948286          0          1 N =   67734
between  .4388518          0          1 n =   11309
within   .2555812  -.5161957   1.372693  T-bar = 5.98939

jobopoverall   .5086826   .4999273          0          1 N =   92138
between  .4050137          0          1 n =   15753
within   .3512974  -.4357618   1.453127  T-bar = 5.84892

tenoverall   4.442097   5.800641          0         50 N =   94382
between  5.086243          0         44 n =   15958
within   3.278007   -33.7579   37.77543  T-bar =  5.9144

soc1xoverall   .1383421   .3452605          0          1 N =   94382
between  .2641729          0          1 n =   15958
within   .2237815  -.8061024   1.082787  T-bar =  5.9144

soc2xoverall   .1010786   .3014344          0          1 N =   94382
between  .2513808          0          1 n =   15958
within   .1622333  -.8433658   1.045523  T-bar =  5.9144

soc3xoverall   .1287216   .3348933          0          1 N =   94382
between  .2749855          0          1 n =   15958
within   .1985793  -.8157229   1.073166  T-bar =  5.9144

soc4xoverall   .1938929   .3953481          0          1 N =   94382
between  .3340848          0          1 n =   15958
within   .2309706  -.7505515   1.138337  T-bar =  5.9144

soc5xoverall   .0963001    .295004          0          1 N =   94382
between  .2653166          0          1 n =   15958
within    .156136  -.8481443   1.040745  T-bar =  5.9144

soc6xoverall   .1198746   .3248165          0          1 N =   94382
between  .3105574          0          1 n =   15958
within   .1775957  -.8245699   1.064319  T-bar =  5.9144

soc7xoverall   .0858109    .280086          0          1 N =   94382
between  .2625724          0          1 n =   15958
within    .178774  -.8586336   1.030255  T-bar =  5.9144

soc8xoverall   .0850692   .2789861          0          1 N =   94382
between  .2463305          0          1 n =   15958
within   .1604164  -.8593753   1.029514  T-bar =  5.9144

soc9xoverall   .0773876   .2672069          0          1 N =   94382
between  .2516964          0          1 n =   15958
within    .164784  -.8670568   1.021832  T-bar =  5.9144

reg1xoverall   .0752673   .2638236          0          1 N =   93706
between  .2571963          0          1 n =   15928
within   .0798445  -.8659091   1.019712  T-bar =  5.8831

reg2xoverall   .1629351   .3693085          0          1 N =   93706
between  .3484894          0          1 n =   15928
within   .0938931  -.7815093    1.10738  T-bar =  5.8831

reg3xoverall   .0755021   .2642012          0          1 N =   93706
between  .2481821          0          1 n =   15928
within   .0605945  -.8475748   1.016679  T-bar =  5.8831

reg4xoverall   .0329968   .1786291          0          1 N =   93706
between  .1652107          0          1 n =   15928
within   .0438604  -.9081797   .9615682  T-bar =  5.8831

reg5xoverall   .0704437   .2558947          0          1 N =   93706
between   .245287          0          1 n =   15928
within   .0615119  -.8740007    1.01162  T-bar =  5.8831

reg6xoverall   .0711481   .2570734          0          1 N =   93706
between   .250747          0          1 n =   15928
within   .0507769  -.8700284   1.004481  T-bar =  5.8831

reg7xoverall   .0863445   .2808736          0          1 N =   93706
between   .263481          0          1 n =   15928
within   .0534598  -.8580999   1.030789  T-bar =  5.8831

reg8xoverall   .0758436   .2647491          0          1 N =   93706
between  .2502664          0          1 n =   15928
within   .0556379  -.8686008   1.020288  T-bar =  5.8831

reg9xoverall   .0514161   .2208462          0          1 N =   93706
between  .2005506          0          1 n =   15928
within   .0406515  -.8819172   .9958606  T-bar =  5.8831

reg10xoverall    .125072   .3308023          0          1 N =   93706
between  .3540098          0          1 n =   15928
within   .0451375   -.812428   1.069516  T-bar =  5.8831

reg11xoverall   .1684417   .3742601          0          1 N =   93706
between   .392804          0          1 n =   15928
within   .0397877  -.7315583   1.109618  T-bar =  5.8831

reg12xoverall   .0045888   .0675856          0          1 N =   93706
between  .0949798          0          1 n =   15928
within          0   .0045888   .0045888  T-bar =  5.8831

hourly~yoverall     5.3808   3.970115   .0204255   240.7125 N =   94382
between  2.977065   .0816397   50.67446 n =   15958
within   2.556224  -39.26834   220.3004  T-bar =  5.9144

logpayoverall   1.518747   .5628849  -3.890971   5.483603 N =   94382
between  .5246163  -2.529048   3.718402 n =   15958
within   .2941827  -2.309889    4.79502  T-bar =  5.9144

femaleoverall   .5240936   .4994218          0          1 N =   94382
between  .4995852          0          1 n =   15958
within          0   .5240936   .5240936  T-bar =  5.9144

sestoverall   .3520798   .4776212          0          1 N =   94382
between  .4127478          0          1 n =   15958
within   .3059767  -.5923646   1.296524  T-bar =  5.9144

mestoverall   .2624547   .4399708          0          1 N =   94382
between  .3512153          0          1 n =   15958
within   .3131856  -.6819897   1.206899  T-bar =  5.9144

lestoverall    .396315   .4891339          0          1 N =   94382
between  .4114489          0          1 n =   15958
within   .2995364  -.5481295   1.340759  T-bar =  5.9144

npermjoverall   .0607001   .2387807          0          1 N =   94382
between  .2455445          0          1 n =   15958
within   .1803257  -.8804763   1.005145  T-bar =  5.9144

parthoverall   .2686635   .4432669          0          1 N =   94382
between  .4083613          0          1 n =   15958
within   .2477528  -.6757809   1.213108  T-bar =  5.9144


. gen ten2 = ten^2

. gen age2 = age^2


. xtreg jbsat7x hourlypay ten ten2 age age2, re robust

Random-effects GLS regression                   Number of obs     =     94,382
Group variable: pid                             Number of groups  =     15,958

R-squared:                                      Obs per group:
     Within  = 0.0023                                         min =          1
     Between = 0.0066                                         avg =        5.9
     Overall = 0.0035                                         max =         18

                                                Wald chi2(5)      =     178.84
corr(u_i, X) = 0 (assumed)                      Prob > chi2       =     0.0000

 clusters in )}

    Robust
     jbsat7x Coefficient  std. err.      z   P>|z|     [95% conf. interval]

hourlypay -.0031034  .001751   -1.770.076-.0065353 .0003284
ten -.0242524 .0025048   -9.680.000-.0291617-.0193431
ten2  .0005991 .0001063    5.630.000 .0003907 .0008075
age -.0203566 .0037777   -5.390.000-.0277608-.0129523
age2  .0003333 .0000492    6.770.000 .0002368 .0004298
_cons  5.573107 .0656116   84.940.000  5.44451 5.701703

     sigma_u   .89019312
     sigma_e   1.1559426
         rho   .37227581   (fraction of variance due to u_i)


. xtreg jbsat7x hourlypay ten ten2 age age2, fe robust

Fixed-effects (within) regressionNumber of obs=    94,382
Group variable: pidNumber of groups=    15,958

R-squared:Obs per group:
     Within  = 0.0037min=         1
     Between = 0.0018avg=       5.9
     Overall = 0.0000max=        18

F(5,15957)=    39.28
corr(u_i, Xb) = -0.1077Prob > F=0.0000

 clusters in )}

    Robust
     jbsat7x Coefficient  std. err.      t   P>|t|     [95% conf. interval]

hourlypay  .0061008 .0016522    3.690.000 .0028622 .0093393
ten -.0331237 .0028418  -11.660.000-.0386939-.0275535
ten2  .0008041 .0001264    6.360.000 .0005564 .0010517
age -.0046956  .005951   -0.790.430-.0163602 .0069691
age2  .0000717 .0000748    0.960.338 -.000075 .0002184
_cons  5.359042 .1147186   46.710.000  5.13418 5.583903

     sigma_u   1.1674454
     sigma_e   1.1559426
         rho   .50495077   (fraction of variance due to u_i)


. xtreg jbsat7x hourlypay ten ten2 age age2, fe

Fixed-effects (within) regressionNumber of obs=    94,382
Group variable: pidNumber of groups=    15,958

R-squared:Obs per group:
     Within  = 0.0037min=         1
     Between = 0.0018avg=       5.9
     Overall = 0.0000max=        18

F(5,78419)=    58.12
corr(u_i, Xb) = -0.1077Prob > F=0.0000


     jbsat7x Coefficient  Std. err.      t   P>|t|     [95% conf. interval]

hourlypay  .0061008  .001515    4.030.000 .0031314 .0090701
ten -.0331237 .0022894  -14.470.000-.0376109-.0286366
ten2  .0008041 .0000967    8.320.000 .0006146 .0009936
age -.0046956 .0042243   -1.110.266-.0129752 .0035841
age2  .0000717 .0000523    1.370.170-.0000308 .0001742
_cons  5.359042  .081262   65.950.000 5.199769 5.518315

     sigma_u   1.1674454
     sigma_e   1.1559426
         rho   .50495077   (fraction of variance due to u_i)

F test that all u_i=0: F(15957, 78419) = 4.18Prob > F = 0.0000

. estimates store FE

. xtreg jbsat7x hourlypay ten ten2 age age2, re

Random-effects GLS regression                   Number of obs     =     94,382
Group variable: pid                             Number of groups  =     15,958

R-squared:                                      Obs per group:
     Within  = 0.0023                                         min =          1
     Between = 0.0066                                         avg =        5.9
     Overall = 0.0035                                         max =         18

                                                Wald chi2(5)      =     284.26
corr(u_i, X) = 0 (assumed)                      Prob > chi2       =     0.0000


     jbsat7x Coefficient  Std. err.      z   P>|z|     [95% conf. interval]

hourlypay -.0031034 .0013449   -2.310.021-.0057394-.0004674
ten -.0242524   .00207  -11.720.000-.0283096-.0201952
ten2  .0005991 .0000852    7.030.000 .0004322  .000766
age -.0203566 .0030906   -6.590.000 -.026414-.0142991
age2  .0003333 .0000395    8.440.000 .0002559 .0004107
_cons  5.573107 .0553867  100.620.000 5.464551 5.681663

     sigma_u   .89019312
     sigma_e   1.1559426
         rho   .37227581   (fraction of variance due to u_i)


. estimates store RE

. hausman FE RE

 Coefficients 
(b)(B)(b-B)sqrt(diag(V_b-V_B))
     FE          RE     DifferenceStd. err.

hourlypay  .0061008-.0031034 .0092042 .0006974
ten -.0331237-.0242524-.0088713 .0009779
ten2  .0008041 .0005991  .000205 .0000458
age -.0046956-.0203566  .015661 .0028798
age2  .0000717 .0003333-.0002616 .0000343

.}
.}

Test of H0: Difference in coefficients not systematic

)} = (b-B)'[(V_b-V_B)^(-1)](b-B)
 = }
 = }


. xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x, fe robust
note:  omitted because of collinearity.
note:  omitted because of collinearity.

Fixed-effects (within) regressionNumber of obs=    93,706
Group variable: pidNumber of groups=    15,928

R-squared:Obs per group:
     Within  = 0.0042min=         1
     Between = 0.0002avg=       5.9
     Overall = 0.0012max=        18

F(15,15927)=    14.24
corr(u_i, Xb) = -0.1058Prob > F=0.0000

 clusters in )}

    Robust
     jbsat7x Coefficient  std. err.      t   P>|t|     [95% conf. interval]

hourlypay   .006425 .0017617    3.650.000 .0029719  .009878
ten -.0332777 .0028518  -11.670.000-.0388675-.0276878
ten2  .0008103 .0001268    6.390.000 .0005617 .0010589
age -.0047423 .0059682   -0.790.427-.0164406  .006956
age2  .0000656  .000075    0.870.382-.0000814 .0002126
reg1x  -.236763 .1398412   -1.690.090-.5108675 .0373416
reg2x -.1306707 .1408102   -0.930.353-.4066746 .1453332
reg3x  .0025988 .1478315    0.020.986-.2871677 .2923653
reg4x  .0190654 .1805358    0.110.916-.3348052 .3729359
reg5x -.0237476 .1457487   -0.160.871-.3094314 .2619363
reg6x -.1957396 .1684548   -1.160.245  -.52593 .1344508
reg7x -.2091361 .1564734   -1.340.181-.5158416 .0975694
reg8x  -.065724 .1481051   -0.440.657-.3560267 .2245788
reg9x  -.327929 .1843045   -1.780.075-.6891867 .0333287
reg10x  .1131479 .1741716    0.650.516-.2282481 .4545439
reg11x         0  (omitted)
reg12x         0  (omitted)
_cons  5.449359 .1549669   35.160.000 5.145606 5.753112

     sigma_u   1.1692402
     sigma_e   1.1553216
         rho   .50598743   (fraction of variance due to u_i)


. xttest0 reg1x reg2x reg3x reg4x reg5x reg6x reg7x reg8x reg9x reg10x reg11x reg12x
last estimates not xtreg, re

. xttest0 reg1x reg2x reg3x reg4x reg5x reg6x reg7x reg8x reg9x reg10x
last estimates not xtreg, re

. xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x, fe
note:  omitted because of collinearity.
note:  omitted because of collinearity.

Fixed-effects (within) regressionNumber of obs=    93,706
Group variable: pidNumber of groups=    15,928

R-squared:Obs per group:
     Within  = 0.0042min=         1
     Between = 0.0002avg=       5.9
     Overall = 0.0012max=        18

F(15,77763)=    21.88
corr(u_i, Xb) = -0.1058Prob > F=0.0000


     jbsat7x Coefficient  Std. err.      t   P>|t|     [95% conf. interval]

hourlypay   .006425 .0015683    4.100.000 .0033511 .0094989
ten -.0332777 .0022953  -14.500.000-.0377764-.0287789
ten2  .0008103 .0000969    8.370.000 .0006205 .0010001
age -.0047423 .0042396   -1.120.263-.0130518 .0035672
age2  .0000656 .0000524    1.250.211-.0000371 .0001684
reg1x  -.236763 .1035203   -2.290.022-.4396621-.0338638
reg2x -.1306707  .101017   -1.290.196-.3286635 .0673221
reg3x  .0025988 .1110427    0.020.981-.2150443 .2202419
reg4x  .0190654 .1274961    0.150.881-.2308262  .268957
reg5x -.0237476 .1106337   -0.210.830-.2405889 .1930938
reg6x -.1957396 .1203447   -1.630.104-.4316145 .0401352
reg7x -.2091361 .1144287   -1.830.068-.4334157 .0151435
reg8x  -.065724  .113241   -0.580.562-.2876757 .1562278
reg9x  -.327929 .1292133   -2.540.011-.5811862-.0746717
reg10x  .1131479 .1251452    0.900.366-.1321361 .3584318
reg11x         0  (omitted)
reg12x         0  (omitted)
_cons  5.449359 .1124021   48.480.000 5.229052 5.669667

     sigma_u   1.1692402
     sigma_e   1.1553216
         rho   .50598743   (fraction of variance due to u_i)

F test that all u_i=0: F(15927, 77763) = 4.15Prob > F = 0.0000


. test reg1x reg2x reg3x reg4x reg5x reg6x reg7x reg8x reg9x reg10x reg11x reg12x

 reg1x = 0
 reg2x = 0
 reg3x = 0
 reg4x = 0
 reg5x = 0
 reg6x = 0
 reg7x = 0
 reg8x = 0
 reg9x = 0
 reg10x = 0
 o.reg11x = 0
 o.reg12x = 0
       Constraint 11 dropped
       Constraint 12 dropped

       F( 10, 77763) =    3.66
Prob > F =    0.0001


. xtreg jbsat7x hourlypay ten ten2 age age2 soc1x-soc9x, fe robust

Fixed-effects (within) regressionNumber of obs=    94,382
Group variable: pidNumber of groups=    15,958

R-squared:Obs per group:
     Within  = 0.0055min=         1
     Between = 0.0004avg=       5.9
     Overall = 0.0023max=        18

F(14,15957)=    21.01
corr(u_i, Xb) = -0.0562Prob > F=0.0000

 clusters in )}

    Robust
     jbsat7x Coefficient  std. err.      t   P>|t|     [95% conf. interval]

hourlypay  .0066701 .0016583    4.020.000 .0034195 .0099206
ten  -.033435 .0028435  -11.760.000-.0390086-.0278614
ten2  .0008094  .000126    6.420.000 .0005624 .0010564
age -.0050891 .0059953   -0.850.396-.0168406 .0066625
age2  .0000811  .000075    1.080.280 -.000066 .0002282
soc1x -.1957131 .0505271   -3.870.000-.2947519-.0966742
soc2x -.1012383 .0504772   -2.010.045-.2001792-.0022974
soc3x -.0106431 .0429937   -0.250.804-.0949156 .0736295
soc4x -.0120704 .0463931   -0.260.795-.1030062 .0788654
soc5x -.0887021  .056639   -1.570.117-.1997208 .0223166
soc6x -.1159981 .0526728   -2.200.028-.2192427-.0127535
soc7x -.1529034  .051133   -2.990.003-.2531298-.0526771
soc8x -.2242477 .0574146   -3.910.000-.3367867-.1117087
soc9x -.1430377 .0550766   -2.600.009 -.250994-.0350815
_cons  5.464014  .124967   43.720.000 5.219064 5.708963

     sigma_u    1.163635
     sigma_e   1.1549551
         rho   .50374356   (fraction of variance due to u_i)


. test soc1x soc2x soc3x soc4x soc5x soc6x soc7x soc8x soc9x

 soc1x = 0
 soc2x = 0
 soc3x = 0
 soc4x = 0
 soc5x = 0
 soc6x = 0
 soc7x = 0
 soc8x = 0
 soc9x = 0

       F(  9, 15957) =   10.87
Prob > F =    0.0000


. xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x soc1x-soc9x if female==0, fe robust
note:  omitted because of collinearity.
note:  omitted because of collinearity.

Fixed-effects (within) regressionNumber of obs=    44,551
Group variable: pidNumber of groups=     7,631

R-squared:Obs per group:
     Within  = 0.0080min=         1
     Between = 0.0081avg=       5.8
     Overall = 0.0084max=        18

F(24,7630)=     7.84
corr(u_i, Xb) = -0.0879Prob > F=0.0000

 clusters in )}

    Robust
     jbsat7x Coefficient  std. err.      t   P>|t|     [95% conf. interval]

hourlypay  .0095417  .002389    3.990.000 .0048586 .0142248
ten -.0304651 .0039604   -7.690.000-.0382286-.0227017
ten2  .0006504  .000164    3.970.000  .000329 .0009718
age -.0241657  .008594   -2.810.005-.0410122-.0073191
age2  .0003792 .0001089    3.480.000 .0001657 .0005926
reg1x -.4325348 .2255819   -1.920.055-.8747373 .0096677
reg2x -.2710237  .231896   -1.170.243-.7256036 .1835562
reg3x -.1734999 .2343423   -0.740.459-.6328752 .2858753
reg4x   -.12585 .2488181   -0.510.613-.6136018 .3619018
reg5x -.0707186 .2273927   -0.310.756-.5164707 .3750336
reg6x -.4197232 .2569584   -1.630.102-.9234323 .0839859
reg7x  -.262022 .2310836   -1.130.257-.7150094 .1909654
reg8x -.0381643 .2455258   -0.160.876-.5194623 .4431337
reg9x -.3903252 .2847995   -1.370.171-.9486105 .1679602
reg10x  .0775001  .257277    0.300.763-.4268335 .5818337
reg11x         0  (omitted)
reg12x         0  (omitted)
soc1x -.1723469 .0767923   -2.240.025 -.322881-.0218128
soc2x  .0093207 .0760946    0.120.903-.1398456 .1584871
soc3x  .0275269 .0675319    0.410.684-.1048543  .159908
soc4x -.0244496 .0707497   -0.350.730-.1631384 .1142392
soc5x -.0392549 .0802289   -0.490.625-.1965257 .1180158
soc6x -.1853919 .0858804   -2.160.031-.3537411-.0170426
soc7x  -.176917 .0818673   -2.160.031-.3373994-.0164346
soc8x -.2050903 .0817663   -2.510.012-.3653747 -.044806
soc9x -.2147825 .0855623   -2.510.012-.3825082-.0470569
_cons  5.678846   .25281   22.460.000 5.183269 6.174423

     sigma_u   1.1953189
     sigma_e    1.156894
         rho   .51633127   (fraction of variance due to u_i)


. xtreg jbsat7x hourlypay ten ten2 age age2 reg1x-reg12x soc1x-soc9x if female==1, fe robust
note:  omitted because of collinearity.
note:  omitted because of collinearity.

Fixed-effects (within) regressionNumber of obs=    49,155
Group variable: pidNumber of groups=     8,297

R-squared:Obs per group:
     Within  = 0.0072min=         1
     Between = 0.0002avg=       5.9
     Overall = 0.0019max=        18

F(24,8296)=     7.99
corr(u_i, Xb) = -0.1354Prob > F=0.0000

 clusters in )}

    Robust
     jbsat7x Coefficient  std. err.      t   P>|t|     [95% conf. interval]

hourlypay  .0046789 .0026949    1.740.083-.0006039 .0099616
ten -.0367904 .0041244   -8.920.000-.0448754-.0287055
ten2  .0010529 .0001975    5.330.000 .0006657 .0014401
age  .0109675  .008391    1.310.191 -.005481  .027416
age2 -.0001892 .0001033   -1.830.067-.0003917 .0000134
reg1x -.0657507 .1616754   -0.410.684-.3826749 .2511736
reg2x -.0039073 .1570566   -0.020.980-.3117775  .303963
reg3x  .1583072 .1802964    0.880.380-.1951187 .5117332
reg4x  .1533631 .2731004    0.560.574-.3819819 .6887081
reg5x  .0227732 .1820482    0.130.900-.3340869 .3796333
reg6x -.0024338 .2113529   -0.010.991-.4167383 .4118708
reg7x -.1568436 .2041746   -0.770.442-.5570768 .2433896
reg8x -.0764314 .1764896   -0.430.665-.4223952 .2695324
reg9x -.3242623 .2258566   -1.440.151-.7669978 .1184732
reg10x  .1232832 .2369389    0.520.603-.3411762 .5877426
reg11x         0  (omitted)
reg12x         0  (omitted)
soc1x -.1898894 .0683794   -2.780.005-.3239302-.0558486
soc2x -.1763364 .0683842   -2.580.010-.3103867-.0422862
soc3x -.0003252 .0564878   -0.010.995-.1110554  .110405
soc4x  .0145707 .0619546    0.240.814-.1068758 .1360172
soc5x -.2731566  .100365   -2.720.007-.4698971 -.076416
soc6x -.0728904 .0681913   -1.070.285-.2065625 .0607816
soc7x -.1209167 .0664742   -1.820.069-.2512227 .0093893
soc8x -.2739977 .0924959   -2.960.003-.4553127-.0926826
soc9x -.0719615  .072239   -1.000.319 -.213568 .0696449
_cons  5.438686  .208357   26.100.000 5.030254 5.847117

     sigma_u   1.1228323
     sigma_e    1.150641
         rho   .48777001   (fraction of variance due to u_i)

. 
end of do-file

. log close

---------------------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Conclusion:

```text
Multiple tests carried out including a Hausman test, guided towards a fixed model choice rather than a random model. Joint significance F-test resulted in favour of keeping regional and occupational dummies in the model due to their non-zero coefficients. Careful analysis of the data implies that women experienced higher satisfaction in terms of hours worked compared to men during the early 1990s. However, females’ extra job satisfaction declined until around 2010, reaching lower levels compared to men. The APS data from 2021 suggests a potential reversal, indicating a possible increase in women's satisfaction levels for unknown reasons. Many results are partially confirming other studies conclusions, e.g., majority of the studies had age and hourly pay as a significant determinant of job satisfaction, whereas in our paper they were only significant for men. An inverse U-shaped relationship between age and job satisfaction for women were noted, perhaps due to their child-bearing nature, with men obtaining approximately twice as much additional satisfaction from a 1 unit increase in hourly pay. However, one could argue that since the coefficient was not statistically significant, age analysis for females could be dropped altogether as found by Chaudhuri(2015). U-shaped relationship between tenure and job satisfaction for both genders was common in literature, and our paper re-confirms this phenomenon. In terms of regional (insignificant) and occupational (partially significant) dummies, the relationship between these and job satisfaction (hours worked) was negative. This suggests negative moods of British people altogether across all industries sampled, implying they would perhaps prefer working fewer hours or more flexible hours. Further studies are recommended showing firms implementing enhanced working conditions in response to negative outcomes of employees in surveys and further research would be recommended to check for workers satisfaction pre and post Covid-19 crisis due to the ‘working from home’ shift.